In [20]:

%pip install -q pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Data preparation
#### Setup

In [ ]:
import numpy as np
import pandas as pd

RANDOM_STATE = 42

# Rutas centralizadas
TRAIN_PATH = "../ames-housing-desarrollo-estudiantes/train.csv"
VAL_PATH = "../ames-housing-desarrollo-estudiantes/validation.csv"

# Cargar datasets
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)

# Separar predictoras y target
X_train = train.drop(columns=["SalePrice", "Id"]).copy()
y_train_raw = train["SalePrice"].copy()

X_val = validation.drop(columns=["SalePrice", "Id"]).copy()
y_val_raw = validation["SalePrice"].copy()

print("\nX_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

Train: (1022, 81)
Validation: (219, 81)

X_train: (1022, 80)
y_train: (1022,)
X_val: (219, 80)
y_val: (219,)


#### Eliminación de features
Por ahora, se eliminará `Id` ya que no representa ninguna característica de la vivienda

In [22]:
X_train = X_train.drop(columns=["Id"])
X_val = X_val.drop(columns=["Id"])

print("Cantidad de features después de eliminar Id:", X_train.shape[1])

Cantidad de features después de eliminar Id: 79


#### Separación de tipos de variables

In [23]:
# Variables nominales
nominal_features = [
    "MSSubClass",
    "MSZoning",
    "Street",
    "Alley",
    "LandContour",
    "Utilities",
    "LotConfig",
    "Neighborhood",
    "Condition1",
    "Condition2",
    "BldgType",
    "HouseStyle",
    "RoofStyle",
    "RoofMatl",
    "Exterior1st",
    "Exterior2nd",
    "MasVnrType",
    "Foundation",
    "Heating",
    "CentralAir",
    "Electrical",
    "Functional",
    "GarageType",
    "PavedDrive",
    "Fence",
    "MiscFeature",
    "SaleType",
    "SaleCondition"
]

# Variables ordinales
ordinal_features = [
    "LotShape",
    "LandSlope",
    "ExterQual",
    "ExterCond",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "HeatingQC",
    "KitchenQual",
    "FireplaceQu",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "PoolQC"
]

# Variables numéricas
numeric_features = [
    col for col in X_train.select_dtypes(include=np.number).columns
    if col not in ["MSSubClass"]
]

print("Numéricas:", len(numeric_features))
print("Nominales:", len(nominal_features))
print("Ordinales:", len(ordinal_features))

Numéricas: 35
Nominales: 28
Ordinales: 16


#### Valores faltantes

`LotFrontage` tiene aproximadamente un 18,2% de faltantes, por lo que no conviene eliminar las filas. Además, el EDA mostró que `LotFrontage` depende bastante del contexto de la vivienda, especialmente del barrio. Por esto,  se puede utilizar la mediana por `Neighborhood`.

In [24]:
# Tomar la mediana solo de train
lotfrontage_medians = (
    X_train
    .groupby("Neighborhood")["LotFrontage"]
    .median()
)

# Aplicar medianas a train
X_train["LotFrontage"] = X_train["LotFrontage"].fillna(
    X_train["Neighborhood"].map(lotfrontage_medians)
)

# Si quedara algún NaN, usar la mediana global de TRAIN
global_median = X_train["LotFrontage"].median()
X_train["LotFrontage"] = X_train["LotFrontage"].fillna(global_median)


# Aplicar las mismas medianas a validation
X_val["LotFrontage"] = X_val["LotFrontage"].fillna(
    X_val["Neighborhood"].map(lotfrontage_medians)
)
X_val["LotFrontage"] = X_val["LotFrontage"].fillna(global_median)

Valores faltantes en variables categóricas que significan ausencia, por lo tanto se los imputa con `None`

In [25]:
categorical_none_features = [
    "PoolQC",
    "MiscFeature",
    "Alley",
    "Fence",
    "FireplaceQu",
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "MasVnrType"
]

for col in categorical_none_features:
    X_train[col] = X_train[col].fillna("None")
    X_val[col] = X_val[col].fillna("None")

Valores faltanes en variables numéricas que significan ausencia, por lo tanto se los imputa con `0`

In [26]:
numeric_none_features = [
    "GarageYrBlt",
    "MasVnrArea"
]

for col in numeric_none_features:
    X_train[col] = X_train[col].fillna(0)
    X_val[col] = X_val[col].fillna(0)


Como `Electrical` tiene solamente 1 valor faltante, se puede utilizar la moda. 

In [27]:
electrical_mode = X_train["Electrical"].mode()[0]

X_train["Electrical"] = X_train["Electrical"].fillna(electrical_mode)
X_val["Electrical"] = X_val["Electrical"].fillna(electrical_mode)

##### Comprobación de valores faltantes imputados correctamente

In [28]:
print("Faltantes en TRAIN:")
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])

print("\nFaltantes en VALIDATION:")
print(X_val.isnull().sum()[X_val.isnull().sum() > 0])

Faltantes en TRAIN:
Series([], dtype: int64)

Faltantes en VALIDATION:
Series([], dtype: int64)


#### Variables categóricas

##### Ordinales: con orden

In [ ]:
ordinal_features = [
    "ExterQual",
    "ExterCond",
    "BsmtQual",
    "BsmtCond",
    "HeatingQC",
    "KitchenQual",
    "FireplaceQu",
    "GarageQual",
    "GarageCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "Functional",
    "GarageFinish",
    "LandSlope",
    "LotShape"
]

##### Nominales: sin orden

In [ ]:
X_train["MSSubClass"] = X_train["MSSubClass"].astype(str)
X_val["MSSubClass"] = X_val["MSSubClass"].astype(str)

In [ ]:
nominal_features = [
    col for col in categorical_features
    if col not in ordinal_features
]

**Aclaración**: originalmente `MSSubClass` contiene enteros, pero no es realmente una variable numérica continua, sino que es un código que representa tipos de vivienda. Es por esto, que termina siendo de tipo categórica nominal.

#### Transformación del target

In [ ]:
y_train = np.log1p(y_train_raw)
y_val = np.log1p(y_val_raw)

print("Skew original:", y_train_raw.skew())
print("Skew transformado:", y_train.skew())

In [ ]:
if X_train["Utilities"].nunique() <= 1:
    X_train = X_train.drop(columns=["Utilities"])
    X_val = X_val.drop(columns=["Utilities"])

Utilities
AllPub    1022
Name: count, dtype: int64
Valores únicos: 1


Como la variable `Utilities` solo tiene un mismo valor, se tomó la decisión de eliminarla debido a que no permite diferenciar casas y no aporta información al modelo.

#### Feature Engineering

In [ ]:
def add_features(df):
    df = df.copy()

    # Superficie total
    df["TotalSF"] = (
        df["TotalBsmtSF"] +
        df["1stFlrSF"] +
        df["2ndFlrSF"]
    )

    # Baños equivalentes
    df["TotalBathrooms"] = (
        df["FullBath"] +
        0.5 * df["HalfBath"] +
        df["BsmtFullBath"] +
        0.5 * df["BsmtHalfBath"]
    )

    # Edad de la vivienda
    df["HouseAgeAtSale"] = (
        df["YrSold"] - df["YearBuilt"]
    )

    # Edad desde remodelación
    df["RemodAgeAtSale"] = (
        df["YrSold"] - df["YearRemodAdd"]
    )

    # Remodelación
    df["IsRemodeled"] = (
        df["YearRemodAdd"] != df["YearBuilt"]
    ).astype(int)

    # Superficie de porches
    df["TotalPorchSF"] = (
        df["OpenPorchSF"] +
        df["3SsnPorch"] +
        df["EnclosedPorch"] +
        df["ScreenPorch"] +
        df["WoodDeckSF"]
    )

    # Indicadores de existencia
    df["HasGarage"] = (df["GarageArea"] > 0).astype(int)
    df["HasBsmt"] = (df["TotalBsmtSF"] > 0).astype(int)
    df["HasFireplace"] = (df["Fireplaces"] > 0).astype(int)
    df["HasPool"] = (df["PoolArea"] > 0).astype(int)
    df["Has2ndFloor"] = (df["2ndFlrSF"] > 0).astype(int)
    df["HasMasVnr"] = (df["MasVnrArea"] > 0).astype(int)

    return df

X_train = add_features(X_train)
X_val = add_features(X_val)